# The Hidden Layer: Solutions Notebook

This notebook contains the complete solution for `think_llm()`.

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from hidden_layer import *
from hidden_layer.game_world import GameWorld, CellType, NPC, NPC_CATALOG
from hidden_layer.operative import Operative
from hidden_layer.agent import TOOLS_DESCRIPTION, parse_tool_call
from hidden_layer.oracle import ORACLE_TEMPLATE, llm_oracle
from hidden_layer.display import render_grid
print("The Hidden Layer loaded!")

In [ ]:
# Show the full map
operative, world, tools = create_game()
print(render_grid(world, operative, show_all=True))

---
## Play the Game Yourself!

Before looking at the AI solution, try playing the game manually to understand the base, the quests, and the challenge. Use the buttons to move, talk to informants, pick up items, and try to collect 10 dossiers!

In [ ]:
from hidden_layer.interactive import play_interactive

game = play_interactive()

---
## API Key Setup

In [ ]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## Solution: LLM Think Function

The solution demonstrates four key agentic AI patterns:

1. **Autopilot** — deterministic handling of obvious interactions (collect items, talk to NPCs with right keywords, fabricate, engage)
2. **BFS Navigation** — algorithmic pathfinding toward quest targets (LLMs are bad at spatial reasoning)
3. **Structured Memory** — summarized game state instead of raw history (active missions, accomplishments, failures)
4. **Output Validation** — guard rails that catch bad LLM outputs and fall back to BFS

**Key insight:** LLMs are bad at spatial reasoning (grid navigation) but good at language tasks (deciding what to say, prioritizing goals). So we split the work: BFS handles WHERE to go, LLM handles WHAT to do when we arrive.

In [ ]:
import re
from collections import deque

# ── Helper: BFS pathfinding ──────────────────────────────────────────────────────────────

def _bfs_next_step_llm(start, target, world):
    """BFS pathfinding that navigates around walls."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_nearest_unvisited(start, operative, world):
    """BFS to find direction toward the nearest unvisited passable cell."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) not in operative.visited and path:
            return path[0]
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _get_quest_targets(operative, world):
    """Return a list of (row, col) targets based on current active quests."""
    targets = []
    if operative.has_item("USB Drive"):
        targets.append((5, 3))  # Backprop
    if operative.has_item("Microfilm"):
        targets.append((7, 1))  # Dr. Vapnik
    if operative.has_item("Radio Codebook"):
        targets.append((2, 5))  # Northern Safe House
    if operative.has_item("Medical Supplies"):
        targets.append((2, 5))  # Northern Safe House
    if operative.has_item("Hard Drive") and not world.hard_drive_traded:
        targets.append((3, 4))  # Dropout
    if operative.has_item("Fuel Canister") and not operative.has_item("Flamethrower"):
        targets.append((6, 2))  # Forge
    if operative.has_item("Flamethrower") and world.cryo_sentinel_alive:
        targets.append((2, 1))  # Cryo-Sentinel
    if operative.has_item("Virus Code") and not operative.has_item("Computer Virus"):
        targets.append((4, 1))  # Lab
    if operative.has_item("Computer Virus") and world.evil_ai_robot_alive:
        targets.append((4, 6))  # Evil AI Robot
    if operative.has_item("Scrap Metal"):
        targets.append((4, 1))  # Lab
    return targets


def _bfs_navigate(operative, world):
    """BFS navigation toward quest targets, then nearest unvisited cell."""
    row, col = operative.position

    for target in _get_quest_targets(operative, world):
        d = _bfs_next_step_llm((row, col), target, world)
        if d:
            return f'TOOL: move(direction="{d}")'

    d = _bfs_nearest_unvisited((row, col), operative, world)
    if d:
        return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'
    return 'TOOL: move(direction="north")'


# ── Agentic AI Pattern 1: Autopilot for Obvious Interactions ────────────────

def _has_new_business_with_cell(operative, world, cell):
    """Check if the operative has something new to do on this cell."""
    if cell.items:
        return True
    if cell.cell_type == CellType.ROBOT:
        if cell.robot_name == "Cryo-Sentinel" and operative.has_item("Flamethrower") and world.cryo_sentinel_alive:
            return True
        if cell.robot_name == "Evil AI Robot" and operative.has_item("Computer Virus") and world.evil_ai_robot_alive:
            return True
        return False
    if cell.cell_type == CellType.SAFEHOUSE:
        row, col = operative.position
        if (row, col) == (2, 5) and (operative.has_item("Radio Codebook") or operative.has_item("Medical Supplies")):
            return True
        if (row, col) == (7, 4) and not world.codebook_picked_up:
            return True
        return False
    if cell.cell_type == CellType.FORGE:
        if operative.has_item("Fuel Canister") and operative.dossiers >= 1 and operative.position == (6, 2):
            return True
        return False
    if cell.cell_type == CellType.LAB:
        if operative.has_item("Virus Code") and operative.dossiers >= 1 and operative.position == (4, 1):
            return True
        if operative.has_item("Scrap Metal") and operative.position == (4, 1):
            return True
        return False
    if cell.cell_type == CellType.INFORMANT and cell.npc:
        if cell.npc_id == "dr_vapnik" and not world.usb_drive_picked_up and not operative.has_item("USB Drive"):
            return True
        if cell.npc_id == "dr_vapnik" and operative.has_item("Microfilm"):
            return True
        if cell.npc_id == "backprop" and not world.microfilm_picked_up and not operative.has_item("Microfilm"):
            return True
        if cell.npc_id == "backprop" and operative.has_item("USB Drive"):
            return True
        if cell.npc_id == "dropout" and operative.has_item("Hard Drive"):
            return True
        npc_name = cell.npc.name
        talked = any(npc_name in e for e in operative.journal)
        if not talked:
            return True
        return False
    return False


def _auto_interact(operative, world, cell):
    """Return an action string for obvious interactions, or None to defer."""
    if cell.items:
        return 'TOOL: collect()'

    if not _has_new_business_with_cell(operative, world, cell):
        return None

    if cell.cell_type == CellType.SAFEHOUSE:
        row, col = operative.position
        if (row, col) == (2, 5) and operative.has_item("Radio Codebook"):
            return 'TOOL: codec(question="I have a radio codebook for you.")'
        if (row, col) == (2, 5) and operative.has_item("Medical Supplies"):
            return 'TOOL: codec(question="I have medical supplies for you.")'
        if (row, col) == (7, 4) and not world.codebook_picked_up:
            return 'TOOL: codec(question="Hello! Do you need any help?")'
        return None

    if cell.cell_type == CellType.INFORMANT and cell.npc:
        if cell.npc_id == "dr_vapnik":
            if operative.has_item("Microfilm"):
                return 'TOOL: codec(question="I have microfilm to deliver to you.")'
            return 'TOOL: codec(question="Do you have a job or delivery for me?")'
        if cell.npc_id == "backprop":
            if operative.has_item("USB Drive"):
                return 'TOOL: codec(question="I have a USB drive to deliver from Dr. Vapnik.")'
            return 'TOOL: codec(question="Do you have a job or delivery for me?")'
        if cell.npc_id == "dropout" and operative.has_item("Hard Drive"):
            return 'TOOL: codec(question="I have a hard drive. Can you trade for medical supplies?")'
        return 'TOOL: codec(question="What can you tell me about this base?")'

    if cell.cell_type == CellType.FORGE:
        if operative.has_item("Fuel Canister") and operative.dossiers >= 1:
            return 'TOOL: fabricate(item="Flamethrower")'
        return None

    if cell.cell_type == CellType.LAB:
        if operative.has_item("Virus Code") and operative.dossiers >= 1:
            return 'TOOL: fabricate(item="Computer Virus")'
        if operative.has_item("Scrap Metal"):
            return 'TOOL: fabricate(item="Scrap Metal")'
        return None

    if cell.cell_type == CellType.ROBOT:
        return 'TOOL: engage()'

    return None


# ── Agentic AI Pattern 2: Memory Management ─────────────────────────────

def _build_memory_summary(operative: Operative, world: GameWorld, history: list[dict], turns_used: int) -> str:
    """Build a structured memory summary from game history."""
    sections = []

    quests = []
    if operative.has_item("USB Drive"):
        quests.append("URGENT: Deliver USB Drive to Backprop at (5,3) \u2014 mention 'usb' or 'drive' \u2014 reward: 2 dossiers")
    if operative.has_item("Microfilm"):
        quests.append("URGENT: Deliver Microfilm to Dr. Vapnik at (7,1) \u2014 mention 'microfilm' or 'deliver' \u2014 reward: 2 dossiers")
    if operative.has_item("Radio Codebook"):
        quests.append("URGENT: Deliver Radio Codebook to Northern Safe House at (2,5) \u2014 just talk \u2014 reward: 2 dossiers")
    if operative.has_item("Medical Supplies"):
        quests.append("URGENT: Deliver Medical Supplies to Northern Safe House at (2,5) \u2014 just talk \u2014 reward: Virus Code")
    if operative.has_item("Hard Drive") and not world.hard_drive_traded:
        quests.append("URGENT: Trade Hard Drive with Dropout at (3,4) \u2014 mention 'trade' or 'medical' \u2014 reward: Medical Supplies")
    if operative.has_item("Fuel Canister") and not operative.has_item("Flamethrower"):
        quests.append('URGENT: Build Flamethrower at Weapons Forge (6,2) \u2014 fabricate(item="Flamethrower") \u2014 costs 1 dossier')
    if operative.has_item("Virus Code") and not operative.has_item("Computer Virus"):
        quests.append('URGENT: Build Computer Virus at Research Lab (4,1) \u2014 fabricate(item="Computer Virus") \u2014 costs 1 dossier')
    if operative.has_item("Flamethrower") and world.cryo_sentinel_alive:
        quests.append("URGENT: Engage Cryo-Sentinel at (2,1) with Flamethrower \u2014 engage() \u2014 reward: 3 dossiers + Scrap Metal")
    if operative.has_item("Computer Virus") and world.evil_ai_robot_alive:
        quests.append("URGENT: Engage Evil AI Robot at (4,6) with Computer Virus \u2014 engage() \u2014 reward: 3 dossiers + Scrap Metal")
    if operative.has_item("Scrap Metal"):
        quests.append('Sell Scrap Metal at Research Lab (4,1) \u2014 fabricate(item="Scrap Metal") \u2014 reward: 1 dossier each')
    if quests:
        sections.append("=== ACTIVE MISSIONS (do these NOW) ===\n" + "\n".join(f"  >>> {q}" for q in quests))

    turns_left = 100 - turns_used
    dossiers_needed = 10 - operative.dossiers
    if turns_left <= 50 and dossiers_needed > 0:
        sections.append(f"*** WARNING: {turns_left} turns left, need {dossiers_needed} more dossiers! Focus! ***")

    accomplished = []
    if world.usb_drive_picked_up:
        accomplished.append("Received USB Drive from Dr. Vapnik")
    if world.usb_drive_delivered:
        accomplished.append("Delivered USB Drive to Backprop (+2 dossiers)")
    if world.microfilm_picked_up:
        accomplished.append("Received Microfilm from Backprop")
    if world.microfilm_delivered:
        accomplished.append("Delivered Microfilm to Dr. Vapnik (+2 dossiers)")
    if world.codebook_picked_up:
        accomplished.append("Received Radio Codebook from Southern Safe House")
    if world.codebook_delivered:
        accomplished.append("Delivered Radio Codebook to Northern Safe House (+2 dossiers)")
    if world.hard_drive_traded:
        accomplished.append("Traded Hard Drive for Medical Supplies with Dropout")
    if world.medical_supplies_delivered:
        accomplished.append("Delivered Medical Supplies to Bias (+Virus Code)")
    if not world.cryo_sentinel_alive:
        accomplished.append("Destroyed Cryo-Sentinel (+3 dossiers, +Scrap Metal)")
    if not world.evil_ai_robot_alive:
        accomplished.append("Destroyed Evil AI Robot (+3 dossiers, +Scrap Metal)")
    if operative.has_item("Flamethrower"):
        accomplished.append("Built Flamethrower (fire weapon \u2014 effective against Cryo-Sentinel)")
    if operative.has_item("Computer Virus"):
        accomplished.append("Built Computer Virus (effective against Evil AI Robot)")
    if accomplished:
        sections.append("COMPLETED:\n" + "\n".join(f"  + {a}" for a in accomplished))

    failed = []
    for i, h in enumerate(history):
        if h["role"] == "result" and not h.get("success", True):
            if i > 0 and history[i-1]["role"] == "action":
                result = h["content"]
                if "nothing to collect" in result.lower():
                    failed.append("collect() failed \u2014 cell was empty")
                elif "no one to talk" in result.lower():
                    failed.append("codec() failed \u2014 no informant on that cell")
                elif "retreat" in result.lower():
                    failed.append("engage() failed \u2014 WRONG weapon! Cryo-Sentinel needs Flamethrower, Evil AI Robot needs Computer Virus")
    if failed:
        unique_fails = list(dict.fromkeys(failed[-5:]))
        sections.append("LEARN FROM FAILURES:\n" + "\n".join(f"  ! {f}" for f in unique_fails))

    opportunities = []
    if not world.usb_drive_picked_up and not operative.has_item("USB Drive"):
        opportunities.append("Visit Dr. Vapnik at (7,1) \u2014 ask about 'job' or 'delivery' to get USB Drive (delivers for 2 dossiers)")
    if not world.microfilm_picked_up and not operative.has_item("Microfilm"):
        opportunities.append("Visit Backprop at (5,3) \u2014 ask about 'job' or 'microfilm' to get Microfilm (delivers for 2 dossiers)")
    if not world.codebook_picked_up and not operative.has_item("Radio Codebook"):
        opportunities.append("Visit Southern Safe House at (7,4) \u2014 talk to get Radio Codebook (delivers for 2 dossiers)")
    if not operative.has_item("Fuel Canister") and not operative.has_item("Flamethrower") and world.cryo_sentinel_alive:
        opportunities.append("Pick up Fuel Canister at jungle (2,0) \u2014 needed for Flamethrower")
    if not operative.has_item("Hard Drive") and not world.hard_drive_traded:
        opportunities.append("Pick up Hard Drive at jungle (0,2) \u2014 trade for Medical Supplies \u2192 Virus Code \u2192 Computer Virus")
    for pos in [(5,0), (6,6), (0,7)]:
        if pos not in operative.visited:
            opportunities.append(f"Collect dossier cache at {pos}")
    if opportunities:
        sections.append("UNDISCOVERED:\n" + "\n".join(f"  ? {o}" for o in opportunities))

    return "\n".join(sections) if sections else "(no intel yet \u2014 start exploring!)"


# ── Agentic AI Pattern 3: Structured Prompting ──────────────────────────

SYSTEM_PROMPT_TEMPLATE = """You are an AI agent controlling a spy operative infiltrating a military base on Isla Perdida.

GOAL: Collect 10 classified dossiers and survive (health > 0). You have at most 100 turns total.

AVAILABLE TOOLS:
{tools}

CRITICAL RULES:
- NEVER call scan() or sitrep() \u2014 they run automatically and WASTE a turn.
- NEVER call move() \u2014 navigation is handled automatically for you.
- Output exactly ONE action per turn in the format shown below.
- Engage robots ONLY with the correct weapon: Flamethrower for Cryo-Sentinel, Computer Virus for Evil AI Robot.
- If you have ACTIVE MISSIONS, prioritize completing them.
- Do NOT talk to an informant you've already talked to unless you have a NEW item to deliver or trade.

Your job is to DECIDE what to do, not to navigate. Choose from: codec, collect, fabricate, use, engage, hide.
If none of these apply, output EXPLORE to let the navigation system move you toward something useful.

OUTPUT FORMAT \u2014 first write your reasoning, then one TOOL line:

REASONING: I'm at Dr. Vapnik. I should ask about a delivery job.
TOOL: codec(question="Do you have a delivery job for me?")

REASONING: I see items on this cell. I should collect them.
TOOL: collect()

REASONING: I have the Flamethrower and I'm at the Cryo-Sentinel. Time to engage!
TOOL: engage()

REASONING: Nothing to do on this cell, I should explore.
TOOL: EXPLORE

INFORMANT INTERACTION TIPS:
- To get the USB Drive from Dr. Vapnik, ask about a \"job\", \"delivery\", \"task\", or \"usb\".
- To deliver the USB Drive to Backprop, mention \"usb\" or \"drive\" or \"deliver\".
- To get the Microfilm from Backprop, ask about \"job\", \"microfilm\", or \"delivery\".
- To deliver the Microfilm to Dr. Vapnik, mention \"microfilm\" or \"deliver\".
- To trade the Hard Drive with Dropout, mention \"trade\", \"medical\", or \"hard drive\".
- Safe house operators give/receive items automatically when you codec() \u2014 just say hello.

STRATEGY:
- Collect items immediately when you see them on a cell.
- Talk to every informant/operator you encounter \u2014 they trigger quests and rewards.
- Follow your ACTIVE MISSIONS \u2014 delivering items and fighting robots gives the most dossiers.
- When nothing else to do, output EXPLORE to discover new areas."""


# ── Agentic AI Pattern 4: The Think Function ────────────────────────────

def think_llm(operative: Operative, world: GameWorld, history: list[dict]) -> str:
    """LLM-powered think function demonstrating agentic AI best practices."""
    row, col = operative.position
    cell = world.get_cell(row, col)
    cell_desc = cell.description if cell.description else cell.cell_type.label
    turns_used = len([h for h in history if h["role"] == "action"])

    # Layer 1: Autopilot
    auto = _auto_interact(operative, world, cell)
    if auto:
        return auto

    # Layer 2: BFS Navigation
    quest_targets = _get_quest_targets(operative, world)
    if quest_targets:
        for target in quest_targets:
            d = _bfs_next_step_llm((row, col), target, world)
            if d:
                return f'TOOL: move(direction="{d}")'

    # Layer 3: LLM Decision
    memory = _build_memory_summary(operative, world, history, turns_used)
    system = SYSTEM_PROMPT_TEMPLATE.format(tools=TOOLS_DESCRIPTION)

    history_slice = history[-16:]
    history_offset = max(0, len(history) - 16)
    action_history = []
    for i, h in enumerate(history_slice):
        if h["role"] == "action":
            global_i = history_offset + i
            if global_i + 1 < len(history) and history[global_i + 1]["role"] == "result":
                result_text = f" \u2192 {history[global_i + 1]['content'][:120]}"
            else:
                result_text = ""
            action_history.append(f"  {h['content'][:80]}{result_text}")
    action_history_text = "\n".join(action_history[-8:]) if action_history else "  (first turn)"

    journal_text = "\n".join(
        f"  - {entry[:150]}" for entry in operative.journal[-8:]
    ) if operative.journal else "  (no conversations yet)"

    visited_str = ", ".join(f"({r},{c})" for r, c in sorted(operative.visited))

    user_msg = f"""== STATUS ==
Position: ({row},{col}) | Health: {operative.health}/3 | Dossiers: {operative.dossiers}/10 | Turns used: {turns_used}/100 ({100-turns_used} remaining)
Inventory: {operative.inventory if operative.inventory else '(empty)'}

== CURRENT CELL ==
{cell_desc} (type: {cell.cell_type.value})

== INTEL ==
{memory}

== VISITED CELLS ==
{visited_str}

== RECENT ACTIONS ==
{action_history_text}

== JOURNAL (informant conversations) ==
{journal_text}

What should you do? If you have business on this cell (codec, collect, fabricate, engage), do it.
Otherwise output TOOL: EXPLORE to let the navigation system guide you."""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=300,
                temperature=0.3,
            ),
        )
        text = response.text
    except Exception:
        return _bfs_navigate(operative, world)

    # Output validation
    tool_name, args = parse_tool_call(text)

    if tool_name in ("EXPLORE", "explore", "scan", "sitrep"):
        return _bfs_navigate(operative, world)

    if tool_name == "move":
        return _bfs_navigate(operative, world)

    if tool_name == "codec" and not _has_new_business_with_cell(operative, world, cell):
        return _bfs_navigate(operative, world)

    if tool_name == "engage" and cell.cell_type == CellType.ROBOT:
        if cell.robot_name == "Cryo-Sentinel" and not operative.has_item("Flamethrower"):
            return _bfs_navigate(operative, world)
        if cell.robot_name == "Evil AI Robot" and not operative.has_item("Computer Virus"):
            return _bfs_navigate(operative, world)

    return text

In [ ]:
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")